# MaskClustering End-to-End Pipeline on Replica

This notebook runs the full MaskClustering pipeline on the Replica dataset, from 2D mask prediction through 3D clustering, semantic labeling, and evaluation.

## Prerequisites

Complete these steps before running any cell:

**1. SSH into the cluster, allocate a GPU node, and activate the environment:**
```bash
ssh <your-username>@ml3d.vc.in.tum.de
salloc --gpus=1
conda activate maskclustering
```

**2. Ensure the Replica dataset is in place inside your MaskClustering directory:**
- Each scene folder (`office0`–`office4`, `room0`–`room2`) must contain: `color/`, `depth/`, `poses/`, `intrinsics.txt`, `{scene}_mesh.ply`
- Ground truth annotations at `data/replica/ground_truth/`

**3. CropFormer model weights:**
- Download `model_final_hornet_3x.pth` and place it at `third_party/detectron2/projects/CropFormer/model_final_hornet_3x.pth`

**4. Launch Jupyter from the MaskClustering root on the compute node:**
```bash
cd /path/to/MaskClustering
jupyter notebook --no-browser --ip=0.0.0.0 --port 8888
```
- `--no-browser`: the cluster has no display, so Jupyter must not try to open one
- `--ip=0.0.0.0`: makes Jupyter accept connections on all interfaces — required for the SSH tunnel to reach it
- `--port 8888`: optional (it is the default), but keeps the tunnel command predictable

**5. Open the SSH tunnel — run Cell 2 first, it will print your exact tunnel command.**
Paste it in a terminal on your local machine, then open `http://localhost:8888`.

---

## Pipeline Overview

| Step | Description |
|------|-------------|
| 0 | 2D mask prediction — generate per-frame instance masks (CropFormer or SAM) |
| 1 | Mask clustering — backproject 2D masks to 3D, build mask graph, run iterative clustering |
| 2 | Class-agnostic evaluation — mAP without semantic labels |
| 3 | CLIP visual feature extraction — extract image embeddings per 3D object instance |
| 4 | CLIP text feature extraction — encode Replica class label names into text embeddings |
| 5 | Semantic label assignment — match visual embeddings to text embeddings per object |
| 6 | Class-aware evaluation — mAP with semantic labels |

In [ ]:
import os
import sys
import subprocess
import socket
import getpass

# ============================================================
# Configuration
# ============================================================
MASK_PREDICTOR = 'cropformer'  # 'cropformer' or 'sam'
scenes = ['office0', 'office1', 'office2', 'office3', 'office4', 'room0', 'room1', 'room2']

# Requires Jupyter to be launched from the MaskClustering root (see prerequisites)
MASKCLUSTERING_ROOT = os.getcwd()
HOSTNAME = socket.gethostname()
USERNAME = getpass.getuser()

os.chdir(MASKCLUSTERING_ROOT)
sys.path.insert(0, MASKCLUSTERING_ROOT)

print(f'Root           : {MASKCLUSTERING_ROOT}')
print(f'Node           : {HOSTNAME}')
print(f'User           : {USERNAME}')
print(f'Mask predictor : {MASK_PREDICTOR}')
print(f'Scenes         : {len(scenes)}')


def run_cmd(cmd, cwd=None):
    """Run a command and stream its output live into the notebook cell."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=cwd if cwd is not None else MASKCLUSTERING_ROOT,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed (exit code {proc.returncode})')

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        f'No GPU found on node "{HOSTNAME}". '
        'Run "salloc --gpus=1" first, then relaunch Jupyter from the MaskClustering root.'
    )

print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print()
print('SSH tunnel — run this on your LOCAL machine to access the notebook:')
print(f'  ssh -L 8888:{HOSTNAME}:8888 {USERNAME}@ml3d.vc.in.tum.de')
print('Then open: http://localhost:8888')

In [ ]:
# Verify data is in place
all_ok = True
for scene in scenes:
    mask_dir = f'data/replica/{scene}/output/mask'
    color_dir = f'data/replica/{scene}/color'
    n_masks = len(os.listdir(mask_dir)) if os.path.exists(mask_dir) else 0
    n_frames = len(os.listdir(color_dir)) if os.path.exists(color_dir) else 0
    has_frames = n_frames == 200
    status = '✓' if has_frames else '✗'
    mask_status = f'{n_masks} masks' if n_masks > 0 else 'no masks (Step 0 will generate them)'
    print(f'{status} {scene}: {n_frames} frames, {mask_status}')
    if not has_frames:
        all_ok = False

print()
print('CropFormer weights:', '✓' if os.path.exists('third_party/detectron2/projects/CropFormer/model_final_hornet_3x.pth') else '✗ MISSING')
print('Text features:', '✓' if os.path.exists('data/text_features/replica.npy') else '(will be generated in Step 4)')
print('Ground truth:', '✓' if os.path.exists('data/replica/ground_truth') else '✗ MISSING')
print()
print('Ready to run!' if all_ok else 'Some data is missing — ensure all scenes have 200 frames.')

## Step 0: 2D Mask Prediction

Generates per-frame instance segmentation masks for all Replica scenes. The mask predictor is controlled by the `MASK_PREDICTOR` variable in Cell 1.

- **CropFormer** (default): Uses EntitySeg HorNet-3x model via detectron2. Produces class-agnostic instance masks.
- **SAM** (TODO): Will replace CropFormer with Segment Anything for improved mask quality.

Output: `data/replica/{scene}/output/mask/*.png` — one mask image per frame, pixel values = instance IDs.

In [ ]:
CROPFORMER_CONFIG = os.path.join(
    'third_party/detectron2/projects/CropFormer/configs',
    'entityv2/entity_segmentation/mask2former_hornet_3x.yaml'
)
CROPFORMER_WEIGHTS = os.path.join(
    'third_party/detectron2/projects/CropFormer',
    'model_final_hornet_3x.pth'
)

if MASK_PREDICTOR == 'cropformer':
    if not os.path.exists(CROPFORMER_CONFIG):
        raise FileNotFoundError(
            f'CropFormer config not found at:\n  {CROPFORMER_CONFIG}\n'
            'Make sure the detectron2/CropFormer submodule is set up correctly.'
        )
    if not os.path.exists(CROPFORMER_WEIGHTS):
        raise FileNotFoundError(
            f'CropFormer weights not found at:\n  {CROPFORMER_WEIGHTS}\n'
            'Download model_final_hornet_3x.pth and place it there.'
        )
    for scene in scenes:
        color_dir = f'data/replica/{scene}/color'
        if not os.path.exists(color_dir):
            raise FileNotFoundError(
                f'Color images missing for {scene}:\n  {color_dir}\n'
                'Upload the Replica dataset first (see prerequisites).'
            )

    # Check completeness: mask count should match frame count
    missing = []
    for s in scenes:
        mask_dir = f'data/replica/{s}/output/mask'
        color_dir = f'data/replica/{s}/color'
        n_expected = len(os.listdir(color_dir))
        n_actual = len(os.listdir(mask_dir)) if os.path.exists(mask_dir) else 0
        if n_actual < n_expected:
            missing.append(s)
            if n_actual > 0:
                print(f'⚠ {s}: only {n_actual}/{n_expected} masks — will re-run CropFormer')

    if not missing:
        print('All masks already exist and are complete, skipping CropFormer.')
    else:
        print(f'Running CropFormer for {len(missing)} scenes: {missing}')
        seq_name_list = '+'.join(missing)
        run_cmd([
            sys.executable,
            'third_party/detectron2/projects/CropFormer/demo_cropformer/mask_predict.py',
            '--config-file', CROPFORMER_CONFIG,
            '--root', 'data/replica',
            '--image_path_pattern', 'color/*.jpg',
            '--dataset', 'replica',
            '--seq_name_list', seq_name_list,
            '--opts', 'MODEL.WEIGHTS', CROPFORMER_WEIGHTS,
        ])

elif MASK_PREDICTOR == 'sam':
    raise NotImplementedError('SAM mask prediction not yet implemented — this is the project improvement target.')

else:
    raise ValueError(f'Unknown MASK_PREDICTOR: {MASK_PREDICTOR!r}. Use "cropformer" or "sam".')

## Step 1: Mask Clustering

Backprojects 2D CropFormer masks to 3D using depth images and camera poses, builds a mask graph via view consensus, and runs iterative clustering to group masks into 3D object instances.

Output: `data/replica/{scene}/output/object/replica/object_dict.npy`

In [ ]:
import torch
from utils.config import get_dataset, update_args
from utils.post_process import post_process
from graph.construction import mask_graph_construction
from graph.iterative_clustering import iterative_clustering
from tqdm import tqdm
import argparse

# Validate: masks must exist before clustering
for scene in scenes:
    mask_dir = f'data/replica/{scene}/output/mask'
    if not os.path.exists(mask_dir) or len(os.listdir(mask_dir)) == 0:
        raise FileNotFoundError(
            f'No masks found for {scene} at:\n  {mask_dir}\n'
            'Run Step 0 first to generate 2D masks.'
        )
    depth_dir = f'data/replica/{scene}/depth'
    if not os.path.exists(depth_dir):
        raise FileNotFoundError(
            f'Depth images missing for {scene}:\n  {depth_dir}\n'
            'Upload the full Replica dataset (color + depth + poses).'
        )
    poses_dir = f'data/replica/{scene}/poses'
    if not os.path.exists(poses_dir):
        raise FileNotFoundError(
            f'Camera poses missing for {scene}:\n  {poses_dir}\n'
            'Upload the full Replica dataset (color + depth + poses).'
        )

def run_clustering(scene, config='replica', debug=False):
    args = argparse.Namespace(seq_name=scene, config=config, debug=debug)
    args = update_args(args)
    dataset = get_dataset(args)
    scene_points = dataset.get_scene_points()
    frame_list = dataset.get_frame_list(args.step)

    with torch.no_grad():
        nodes, observer_num_thresholds, mask_point_clouds, point_frame_matrix = mask_graph_construction(args, scene_points, frame_list, dataset)
        object_list = iterative_clustering(nodes, observer_num_thresholds, args.view_consensus_threshold, args.debug)
        post_process(dataset, object_list, mask_point_clouds, scene_points, point_frame_matrix, frame_list, args)

for scene in tqdm(scenes, desc='Clustering scenes'):
    print(f'\n--- Processing {scene} ---')
    run_clustering(scene)

## Step 2: Class-Agnostic Evaluation

Evaluates 3D instance segmentation quality without semantic labels.

In [ ]:
# Validate: clustering output must exist
pred_dir = 'data/prediction/replica_class_agnostic'
if not os.path.exists(pred_dir) or len(os.listdir(pred_dir)) == 0:
    raise FileNotFoundError(
        f'No class-agnostic predictions found at:\n  {pred_dir}\n'
        'Run Step 1 (Mask Clustering) first.'
    )
if not os.path.exists('data/replica/ground_truth'):
    raise FileNotFoundError(
        'Ground truth not found at:\n  data/replica/ground_truth/\n'
        'Upload the Replica ground truth annotations.'
    )

run_cmd([
    sys.executable, '-m', 'evaluation.evaluate',
    '--pred_path', pred_dir,
    '--gt_path', 'data/replica/ground_truth',
    '--dataset', 'replica',
    '--no_class',
])

## Step 3: CLIP Visual Feature Extraction

For each 3D object instance, crops its 2D appearances across frames at multiple scales and extracts CLIP ViT-H-14 visual embeddings.

Output: `data/replica/{scene}/output/object/replica/open-vocabulary_features.npy`

In [ ]:
# Validate: object_dict must exist (from Step 1 clustering)
for scene in scenes:
    obj_dict = f'data/replica/{scene}/output/object/replica/object_dict.npy'
    if not os.path.exists(obj_dict):
        raise FileNotFoundError(
            f'Clustering output missing for {scene}:\n  {obj_dict}\n'
            'Run Step 1 (Mask Clustering) first.'
        )

seq_name_list = '+'.join(scenes)
run_cmd([
    sys.executable, '-m', 'semantics.get_open-voc_features',
    '--config', 'replica',
    '--seq_name_list', seq_name_list,
])

## Step 4: CLIP Text Feature Extraction

Encodes Replica class label names into CLIP ViT-H-14 text embeddings. These are matched against the visual embeddings in Step 5.

Output: `data/text_features/replica.npy`

In [ ]:
text_features_path = 'data/text_features/replica.npy'
if os.path.exists(text_features_path):
    print(f'Text features already exist at {text_features_path}, skipping.')
else:
    run_cmd([sys.executable, '-m', 'semantics.extract_label_featrues'])

## Step 5: Semantic Label Assignment

Matches each object's CLIP visual embedding to the nearest Replica class label text embedding.

Output: `data/prediction/replica/{scene}.npz`

In [ ]:
# Validate: visual features and text features must exist
for scene in scenes:
    feat_path = f'data/replica/{scene}/output/object/replica/open-vocabulary_features.npy'
    if not os.path.exists(feat_path):
        raise FileNotFoundError(
            f'CLIP visual features missing for {scene}:\n  {feat_path}\n'
            'Run Step 3 (CLIP Visual Feature Extraction) first.'
        )
if not os.path.exists('data/text_features/replica.npy'):
    raise FileNotFoundError(
        'CLIP text features missing:\n  data/text_features/replica.npy\n'
        'Run Step 4 (CLIP Text Feature Extraction) first.'
    )

from tqdm import tqdm
for scene in tqdm(scenes, desc='Semantic label assignment'):
    print(f'\n--- {scene} ---')
    run_cmd([
        sys.executable, '-m', 'semantics.open-voc_query',
        '--config', 'replica',
        '--seq_name', scene,
    ])

## Step 6: Class-Aware Evaluation

Evaluates both instance segmentation quality and semantic label accuracy. This is the **baseline mAP** for the project.

In [ ]:
# Validate: semantic predictions must exist
pred_dir = 'data/prediction/replica'
if not os.path.exists(pred_dir) or len(os.listdir(pred_dir)) == 0:
    raise FileNotFoundError(
        f'No semantic predictions found at:\n  {pred_dir}\n'
        'Run Step 5 (Semantic Label Assignment) first.'
    )

run_cmd([
    sys.executable, '-m', 'evaluation.evaluate',
    '--pred_path', pred_dir,
    '--gt_path', 'data/replica/ground_truth',
    '--dataset', 'replica',
])

## Visualization

Visualize the 3D class-agnostic instance segmentation for a single scene using Pyviz3D.

In [ ]:
scene_to_visualize = 'office0'

# Validate: predictions must exist for the chosen scene
vis_pred = f'data/prediction/replica_class_agnostic/{scene_to_visualize}.npz'
if not os.path.exists(vis_pred):
    raise FileNotFoundError(
        f'No predictions found for visualization:\n  {vis_pred}\n'
        'Run Steps 1-2 first, or change scene_to_visualize to a scene that has been processed.'
    )
mesh_path = f'data/replica/{scene_to_visualize}/{scene_to_visualize}_mesh.ply'
if not os.path.exists(mesh_path):
    raise FileNotFoundError(
        f'Mesh file missing:\n  {mesh_path}\n'
        'Make sure the Replica dataset includes the .ply mesh for this scene.'
    )

run_cmd([
    sys.executable, '-m', 'visualize.vis_scene',
    '--config', 'replica',
    '--seq_name', scene_to_visualize,
])